In [1]:
# ============================================================
# ZapKart Dark Store Intelligence
# Notebook 2: Data Cleaning & Preprocessing
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

# Paste your exact same folder path here
RAW_PATH       = "D:\Projects\End-to-end projects\9. ZapKart Dark-Store Intelligence\Data\Raw" + "\\"
PROCESSED_PATH = "D:\Projects\End-to-end projects\9. ZapKart Dark-Store Intelligence\Data\Processed" + "\\"
os.makedirs(PROCESSED_PATH, exist_ok=True)

# Audit log — every cleaning action gets recorded here
cleaning_log = []

def log_action(table, issue, action, rows_affected):
    cleaning_log.append({
        'table':         table,
        'issue_found':   issue,
        'action_taken':  action,
        'rows_affected': rows_affected
    })
    print(f"   📋 LOG: [{table}] {issue} → {action} ({rows_affected} rows)")

print("✅ Setup complete")
print(f"✅ Raw path:       {RAW_PATH}")
print(f"✅ Processed path: {PROCESSED_PATH}")

✅ Setup complete
✅ Raw path:       D:\Projects\End-to-end projects\9. ZapKart Dark-Store Intelligence\Data\Raw\
✅ Processed path: D:\Projects\End-to-end projects\9. ZapKart Dark-Store Intelligence\Data\Processed\


In [2]:
# ============================================================
# Load all raw datasets
# ============================================================

dim_stores           = pd.read_csv(f"{RAW_PATH}dim_stores.csv")
dim_skus             = pd.read_csv(f"{RAW_PATH}dim_skus.csv")
dim_customers        = pd.read_csv(f"{RAW_PATH}dim_customers.csv")
dim_pickers          = pd.read_csv(f"{RAW_PATH}dim_pickers.csv")
fact_orders          = pd.read_csv(f"{RAW_PATH}fact_orders.csv",
                                    parse_dates=['order_timestamp', 'order_date'])
fact_order_items     = pd.read_csv(f"{RAW_PATH}fact_order_items.csv",
                                    parse_dates=['order_date'])
fact_picker_activity = pd.read_csv(f"{RAW_PATH}fact_picker_activity.csv",
                                    parse_dates=['pick_start_time',
                                                 'pick_end_time', 'order_date'])
fact_substitutions   = pd.read_csv(f"{RAW_PATH}fact_substitutions.csv",
                                    parse_dates=['order_date'])

all_tables = {
    'dim_stores':           dim_stores,
    'dim_skus':             dim_skus,
    'dim_customers':        dim_customers,
    'dim_pickers':          dim_pickers,
    'fact_orders':          fact_orders,
    'fact_order_items':     fact_order_items,
    'fact_picker_activity': fact_picker_activity,
    'fact_substitutions':   fact_substitutions
}

print("=" * 60)
print("RAW DATA LOAD SUMMARY")
print("=" * 60)
for name, df in all_tables.items():
    print(f"  {name:<30} {df.shape[0]:>10,} rows  {df.shape[1]:>3} cols")
print("=" * 60)

RAW DATA LOAD SUMMARY
  dim_stores                             10 rows   12 cols
  dim_skus                              499 rows   14 cols
  dim_customers                      50,000 rows   12 cols
  dim_pickers                            80 rows    9 cols
  fact_orders                       453,036 rows   28 cols
  fact_order_items                2,718,037 rows   12 cols
  fact_picker_activity              412,015 rows   12 cols
  fact_substitutions                 54,396 rows   12 cols


In [3]:
# ============================================================
# Profile every table: nulls, dtypes, duplicates, ranges
# ============================================================

def profile_table(name, df):
    print(f"\n{'='*60}")
    print(f"PROFILING: {name}")
    print(f"{'='*60}")
    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

    nulls    = df.isnull().sum()
    null_pct = (nulls / len(df) * 100).round(2)
    null_df  = pd.DataFrame({
        'Null Count': nulls,
        'Null %':     null_pct,
        'Dtype':      df.dtypes
    })
    null_df = null_df[null_df['Null Count'] > 0]
    if len(null_df) > 0:
        print(f"\n WARNING - NULLS FOUND:")
        print(null_df.to_string())
    else:
        print(f"\n✅ No nulls found")

    dupes = df.duplicated().sum()
    if dupes > 0:
        print(f"\n WARNING - DUPLICATES: {dupes:,} duplicate rows found")
    else:
        print(f"✅ No duplicate rows")

    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if num_cols:
        print(f"\n NUMERIC RANGES:")
        print(df[num_cols].describe().round(2).to_string())

for name, df in all_tables.items():
    profile_table(name, df)


PROFILING: dim_stores
Shape: 10 rows x 12 columns

✅ No nulls found
✅ No duplicate rows

 NUMERIC RANGES:
       latitude  longitude  store_area_sqft  max_picker_capacity  monthly_fixed_cost_inr  catchment_radius_km
count     10.00      10.00            10.00                10.00                   10.00                10.00
mean      21.05      76.04          2910.00                 8.90               520000.00                 2.69
std        6.95       2.20           299.81                 1.20                76485.29                 0.28
min       12.94      72.83          2500.00                 7.00               410000.00                 2.30
25%       14.50      73.95          2725.00                 8.00               486250.00                 2.50
50%       19.12      77.22          2850.00                 9.00               502500.00                 2.65
75%       28.56      77.53          3075.00                 9.75               535000.00                 2.88
max       28.

In [4]:
# ============================================================
# CLEAN: dim_stores
# ============================================================

print("CLEANING: dim_stores")
print("-" * 40)

df = dim_stores.copy()
original_rows = len(df)

df['launch_date'] = pd.to_datetime(df['launch_date'])

india_lat_bounds = (8.0, 37.0)
india_lon_bounds = (68.0, 97.0)
invalid_coords   = df[
    ~df['latitude'].between(*india_lat_bounds) |
    ~df['longitude'].between(*india_lon_bounds)
]
if len(invalid_coords) > 0:
    log_action('dim_stores', 'Invalid coordinates',
               'Flagged for review', len(invalid_coords))
else:
    print("   ✅ All coordinates within India bounds")

neg_cost = df[df['monthly_fixed_cost_inr'] <= 0]
if len(neg_cost) > 0:
    df = df[df['monthly_fixed_cost_inr'] > 0]
    log_action('dim_stores', 'Negative fixed cost',
               'Rows removed', len(neg_cost))
else:
    print("   ✅ All fixed costs are positive")

df['store_age_days'] = (pd.Timestamp.today() - df['launch_date']).dt.days
df['cost_per_sqft']  = (df['monthly_fixed_cost_inr'] /
                         df['store_area_sqft']).round(2)
df['city']           = df['city'].str.strip().str.title()

log_action('dim_stores', 'Added derived columns',
           'store_age_days, cost_per_sqft added', len(df))

dim_stores_clean = df
dim_stores_clean.to_csv(f"{PROCESSED_PATH}dim_stores_clean.csv", index=False)
print(f"\n✅ dim_stores cleaned: {len(dim_stores_clean)} rows saved")

CLEANING: dim_stores
----------------------------------------
   ✅ All coordinates within India bounds
   ✅ All fixed costs are positive
   📋 LOG: [dim_stores] Added derived columns → store_age_days, cost_per_sqft added (10 rows)

✅ dim_stores cleaned: 10 rows saved


In [5]:
# ============================================================
# CLEAN: dim_skus
# ============================================================

print("CLEANING: dim_skus")
print("-" * 40)

df = dim_skus.copy()

zero_price = df[df['unit_price_inr'] <= 0]
if len(zero_price) > 0:
    df = df[df['unit_price_inr'] > 0]
    log_action('dim_skus', 'Zero/negative unit price',
               'Rows removed', len(zero_price))
else:
    print("   ✅ All prices are positive")

price_anomaly = df[df['cost_price_inr'] >= df['unit_price_inr']]
if len(price_anomaly) > 0:
    df.loc[df['cost_price_inr'] >= df['unit_price_inr'],
           'cost_price_inr'] = df['unit_price_inr'] * 0.80
    log_action('dim_skus', 'Cost >= Selling price',
               'Cost set to 80% of selling price', len(price_anomaly))
else:
    print("   ✅ All cost prices are below selling prices")

mrp_anomaly = df[df['mrp_inr'] < df['unit_price_inr']]
if len(mrp_anomaly) > 0:
    df.loc[df['mrp_inr'] < df['unit_price_inr'],
           'mrp_inr'] = df['unit_price_inr'] * 1.10
    log_action('dim_skus', 'MRP < selling price',
               'MRP set to 110% of selling price', len(mrp_anomaly))
else:
    print("   ✅ All MRP values are valid")

df['gross_margin_pct_actual'] = ((df['unit_price_inr'] -
                                   df['cost_price_inr']) /
                                   df['unit_price_inr']).round(4)
df['category']       = df['category'].str.strip()
df['velocity_class'] = df['velocity_class'].str.strip().str.upper()
df['sku_name']       = df['sku_name'].str.strip()
df['is_high_margin'] = df['gross_margin_pct_actual'] > 0.30

invalid_shelf = df[df['shelf_life_days'] <= 0]
if len(invalid_shelf) > 0:
    df = df[df['shelf_life_days'] > 0]
    log_action('dim_skus', 'Invalid shelf life',
               'Rows removed', len(invalid_shelf))

log_action('dim_skus', 'Derived columns added',
           'gross_margin_pct_actual, is_high_margin added', len(df))

dim_skus_clean = df
dim_skus_clean.to_csv(f"{PROCESSED_PATH}dim_skus_clean.csv", index=False)
print(f"\n✅ dim_skus cleaned: {len(dim_skus_clean)} rows saved")
print(f"   High margin SKUs: {df['is_high_margin'].sum()}")

CLEANING: dim_skus
----------------------------------------
   ✅ All prices are positive
   ✅ All cost prices are below selling prices
   ✅ All MRP values are valid
   📋 LOG: [dim_skus] Derived columns added → gross_margin_pct_actual, is_high_margin added (499 rows)

✅ dim_skus cleaned: 499 rows saved
   High margin SKUs: 90


In [6]:
# ============================================================
# CLEAN: dim_customers
# ============================================================

print("CLEANING: dim_customers")
print("-" * 40)

df = dim_customers.copy()

df['signup_date'] = pd.to_datetime(df['signup_date'])

future_signups = df[df['signup_date'] > pd.Timestamp.today()]
if len(future_signups) > 0:
    df = df[df['signup_date'] <= pd.Timestamp.today()]
    log_action('dim_customers', 'Future signup dates',
               'Rows removed', len(future_signups))
else:
    print("   ✅ All signup dates are valid")

neg_basket = df[df['avg_basket_size_inr'] <= 0]
if len(neg_basket) > 0:
    df = df[df['avg_basket_size_inr'] > 0]
    log_action('dim_customers', 'Zero/negative basket size',
               'Rows removed', len(neg_basket))

p99 = df['avg_basket_size_inr'].quantile(0.99)
extreme_basket = df[df['avg_basket_size_inr'] > p99]
if len(extreme_basket) > 0:
    df.loc[df['avg_basket_size_inr'] > p99,
           'avg_basket_size_inr'] = p99
    log_action('dim_customers', f'Extreme basket size > {p99:.0f}',
               'Capped at 99th percentile', len(extreme_basket))

neg_orders = df[df['lifetime_orders'] < 0]
if len(neg_orders) > 0:
    df.loc[df['lifetime_orders'] < 0, 'lifetime_orders'] = 0
    log_action('dim_customers', 'Negative lifetime orders',
               'Set to 0', len(neg_orders))

df['tenure_days']      = (pd.Timestamp.today() -
                           df['signup_date']).dt.days
df['city']             = df['city'].str.strip().str.title()
df['customer_segment'] = df['customer_segment'].str.strip()
df['email']            = df['email'].str.lower().str.strip()

dupe_emails = df.duplicated(subset=['email'], keep='first').sum()
if dupe_emails > 0:
    df = df.drop_duplicates(subset=['email'], keep='first')
    log_action('dim_customers', 'Duplicate emails',
               'Kept first occurrence', dupe_emails)
else:
    print("   ✅ No duplicate emails found")

log_action('dim_customers', 'Derived columns added',
           'tenure_days added', len(df))

dim_customers_clean = df
dim_customers_clean.to_csv(
    f"{PROCESSED_PATH}dim_customers_clean.csv", index=False)
print(f"\n✅ dim_customers cleaned: {len(dim_customers_clean):,} rows saved")

CLEANING: dim_customers
----------------------------------------
   ✅ All signup dates are valid
   📋 LOG: [dim_customers] Extreme basket size > 985 → Capped at 99th percentile (500 rows)
   📋 LOG: [dim_customers] Duplicate emails → Kept first occurrence (2390 rows)
   📋 LOG: [dim_customers] Derived columns added → tenure_days added (47610 rows)

✅ dim_customers cleaned: 47,610 rows saved


In [7]:
# ============================================================
# CLEAN: dim_pickers
# ============================================================

print("CLEANING: dim_pickers")
print("-" * 40)

df = dim_pickers.copy()

df['join_date'] = pd.to_datetime(df['join_date'])

future_joins = df[df['join_date'] > pd.Timestamp.today()]
if len(future_joins) > 0:
    df = df[df['join_date'] <= pd.Timestamp.today()]
    log_action('dim_pickers', 'Future join dates',
               'Rows removed', len(future_joins))
else:
    print("   ✅ All join dates valid")

salary_anomaly = df[~df['monthly_salary_inr'].between(10000, 35000)]
if len(salary_anomaly) > 0:
    df.loc[~df['monthly_salary_inr'].between(10000, 35000),
           'monthly_salary_inr'] = df['monthly_salary_inr'].median()
    log_action('dim_pickers', 'Salary out of range',
               'Replaced with median salary', len(salary_anomaly))
else:
    print("   ✅ All salaries within valid range")

valid_store_ids = dim_stores_clean['store_id'].tolist()
invalid_store   = df[~df['store_id'].isin(valid_store_ids)]
if len(invalid_store) > 0:
    df = df[df['store_id'].isin(valid_store_ids)]
    log_action('dim_pickers', 'Invalid store_id FK',
               'Rows removed', len(invalid_store))
else:
    print("   ✅ All store_id foreign keys valid")

df['experience_months'] = ((pd.Timestamp.today() -
                             df['join_date']).dt.days // 30)
df['annual_cost_inr']   = df['monthly_salary_inr'] * 12

log_action('dim_pickers',
           'Recalculated experience_months from join_date',
           'experience_months updated', len(df))

dim_pickers_clean = df
dim_pickers_clean.to_csv(
    f"{PROCESSED_PATH}dim_pickers_clean.csv", index=False)
print(f"\n✅ dim_pickers cleaned: {len(dim_pickers_clean)} rows saved")
print(dim_pickers_clean.groupby('skill_level')['picker_id'].count())

CLEANING: dim_pickers
----------------------------------------
   ✅ All join dates valid
   ✅ All salaries within valid range
   ✅ All store_id foreign keys valid
   📋 LOG: [dim_pickers] Recalculated experience_months from join_date → experience_months updated (80 rows)

✅ dim_pickers cleaned: 80 rows saved
skill_level
Beginner        15
Expert          22
Intermediate    43
Name: picker_id, dtype: int64


In [8]:
# ============================================================
# CLEAN: fact_orders
# ============================================================

print("CLEANING: fact_orders")
print("-" * 40)

df            = fact_orders.copy()
original_rows = len(df)

dupe_orders = df.duplicated(subset=['order_id']).sum()
if dupe_orders > 0:
    df = df.drop_duplicates(subset=['order_id'], keep='first')
    log_action('fact_orders', 'Duplicate order_ids',
               'Kept first occurrence', dupe_orders)
else:
    print("   ✅ No duplicate order_ids")

zero_value = df[df['order_value_inr'] <= 0]
if len(zero_value) > 0:
    df = df[df['order_value_inr'] > 0]
    log_action('fact_orders', 'Zero/negative order value',
               'Rows removed', len(zero_value))
else:
    print("   ✅ All order values positive")

time_cols = ['pick_time_mins', 'pack_time_mins',
             'dispatch_time_mins', 'travel_time_mins']
for col in time_cols:
    neg_time = df[df[col] < 0]
    if len(neg_time) > 0:
        df.loc[df[col] < 0, col] = df[col].median()
        log_action('fact_orders', f'Negative {col}',
                   'Replaced with median', len(neg_time))

df['total_delivery_mins_recalc'] = (df['pick_time_mins'] +
                                     df['pack_time_mins'] +
                                     df['dispatch_time_mins'] +
                                     df['travel_time_mins'])

discrepancy = ((df['total_delivery_mins'] -
                df['total_delivery_mins_recalc']).abs() > 0.5).sum()
if discrepancy > 0:
    df['total_delivery_mins'] = df['total_delivery_mins_recalc']
    log_action('fact_orders',
               f'Delivery time discrepancy in {discrepancy} rows',
               'Recalculated from components', discrepancy)
else:
    print("   ✅ Delivery time components consistent")
df.drop(columns=['total_delivery_mins_recalc'], inplace=True)

df['sla_met'] = df['total_delivery_mins'] <= 10
log_action('fact_orders', 'SLA flag recalculated',
           'Based on total_delivery_mins <= 10', len(df))

extreme_delivery = df[df['total_delivery_mins'] > 120]
if len(extreme_delivery) > 0:
    df = df[df['total_delivery_mins'] <= 120]
    log_action('fact_orders', 'Delivery time > 120 mins',
               'Rows removed as data error', len(extreme_delivery))

bad_discount = df[df['discount_amount_inr'] >= df['order_value_inr']]
if len(bad_discount) > 0:
    df.loc[df['discount_amount_inr'] >= df['order_value_inr'],
           'discount_amount_inr'] = 0
    log_action('fact_orders', 'Discount >= order value',
               'Discount set to 0', len(bad_discount))

valid_stores    = dim_stores_clean['store_id'].tolist()
invalid_store_fk = df[~df['store_id'].isin(valid_stores)]
if len(invalid_store_fk) > 0:
    df = df[df['store_id'].isin(valid_stores)]
    log_action('fact_orders', 'Invalid store_id FK',
               'Rows removed', len(invalid_store_fk))
else:
    print("   ✅ All store_id FKs valid")

df['gross_profit_inr']    = (df['net_order_value_inr'] -
                              df['delivery_cost_inr']).round(2)
df['is_profitable']       = df['gross_profit_inr'] > 0
df['order_hour_band']     = pd.cut(
    df['order_hour'],
    bins=[0, 6, 12, 17, 21, 24],
    labels=['Late Night', 'Morning', 'Afternoon', 'Evening', 'Night'],
    right=False)
df['delivery_speed_band'] = pd.cut(
    df['total_delivery_mins'],
    bins=[0, 8, 10, 15, 25, 120],
    labels=['<8 mins', '8-10 mins', '10-15 mins',
            '15-25 mins', '>25 mins'],
    right=False)

log_action('fact_orders', 'Derived columns added',
           'gross_profit_inr, is_profitable, bands added', len(df))

fact_orders_clean = df
fact_orders_clean.to_csv(
    f"{PROCESSED_PATH}fact_orders_clean.csv", index=False)

print(f"\n✅ fact_orders cleaned")
print(f"   Original rows:     {original_rows:,}")
print(f"   Cleaned rows:      {len(fact_orders_clean):,}")
print(f"   Rows removed:      {original_rows - len(fact_orders_clean):,}")
print(f"   SLA Met %:         {fact_orders_clean['sla_met'].mean()*100:.1f}%")
print(f"   Profitable orders: {fact_orders_clean['is_profitable'].mean()*100:.1f}%")

CLEANING: fact_orders
----------------------------------------
   ✅ No duplicate order_ids
   ✅ All order values positive
   📋 LOG: [fact_orders] Delivery time discrepancy in 79875 rows → Recalculated from components (79875 rows)
   📋 LOG: [fact_orders] SLA flag recalculated → Based on total_delivery_mins <= 10 (453036 rows)
   ✅ All store_id FKs valid
   📋 LOG: [fact_orders] Derived columns added → gross_profit_inr, is_profitable, bands added (453036 rows)

✅ fact_orders cleaned
   Original rows:     453,036
   Cleaned rows:      453,036
   Rows removed:      0
   SLA Met %:         4.5%
   Profitable orders: 100.0%


In [9]:
# ============================================================
# CLEAN: fact_order_items, fact_picker_activity,
#        fact_substitutions
# ============================================================

# --- fact_order_items ---
print("CLEANING: fact_order_items")
print("-" * 40)

df       = fact_order_items.copy()
original = len(df)

df = df[df['quantity'] > 0]
df = df[df['unit_price_inr'] > 0]

df['item_total_inr']  = (df['unit_price_inr'] *
                          df['quantity']).round(2)
df['item_margin_inr'] = ((df['unit_price_inr'] -
                           df['unit_cost_inr']) *
                           df['quantity']).round(2)

valid_orders  = fact_orders_clean['order_id'].tolist()
orphan_items  = df[~df['order_id'].isin(valid_orders)]
if len(orphan_items) > 0:
    df = df[df['order_id'].isin(valid_orders)]
    log_action('fact_order_items', 'Orphan items',
               'Rows removed', len(orphan_items))
else:
    print("   ✅ All order_id FKs valid")

df['item_margin_pct'] = (df['item_margin_inr'] /
                          df['item_total_inr']).round(4)

fact_order_items_clean = df
fact_order_items_clean.to_csv(
    f"{PROCESSED_PATH}fact_order_items_clean.csv", index=False)
print(f"✅ fact_order_items: {original:,} → {len(df):,} rows\n")

# --- fact_picker_activity ---
print("CLEANING: fact_picker_activity")
print("-" * 40)

df       = fact_picker_activity.copy()
original = len(df)

df = df[df['pick_duration_mins'] > 0]

extreme_rate = df[df['pick_rate_per_min'] > 20]
if len(extreme_rate) > 0:
    df.loc[df['pick_rate_per_min'] > 20, 'pick_rate_per_min'] = 20
    log_action('fact_picker_activity', 'Pick rate > 20 items/min',
               'Capped at 20', len(extreme_rate))

valid_pickers = dim_pickers_clean['picker_id'].tolist()
invalid_pk    = df[~df['picker_id'].isin(valid_pickers)]
if len(invalid_pk) > 0:
    df = df[df['picker_id'].isin(valid_pickers)]
    log_action('fact_picker_activity', 'Invalid picker_id FK',
               'Rows removed', len(invalid_pk))
else:
    print("   ✅ All picker_id FKs valid")

df['productivity_band'] = pd.cut(
    df['pick_rate_per_min'],
    bins=[0, 0.8, 1.2, 1.8, 20],
    labels=['Low', 'Medium', 'High', 'Elite'])

fact_picker_activity_clean = df
fact_picker_activity_clean.to_csv(
    f"{PROCESSED_PATH}fact_picker_activity_clean.csv", index=False)
print(f"✅ fact_picker_activity: {original:,} → {len(df):,} rows\n")

# --- fact_substitutions ---
print("CLEANING: fact_substitutions")
print("-" * 40)

df       = fact_substitutions.copy()
original = len(df)

same_sku = df[df['original_sku_id'] == df['substitute_sku_id']]
if len(same_sku) > 0:
    df = df[df['original_sku_id'] != df['substitute_sku_id']]
    log_action('fact_substitutions', 'Original == Substitute SKU',
               'Rows removed', len(same_sku))
else:
    print("   ✅ No self-substitutions found")

valid_orders = fact_orders_clean['order_id'].tolist()
orphan_subs  = df[~df['order_id'].isin(valid_orders)]
if len(orphan_subs) > 0:
    df = df[df['order_id'].isin(valid_orders)]
    log_action('fact_substitutions', 'Orphan substitutions',
               'Rows removed', len(orphan_subs))
else:
    print("   ✅ All order_id FKs valid")

df['acceptance_label'] = df['customer_accepted'].map(
    {True: 'Accepted', False: 'Rejected'})

fact_substitutions_clean = df
fact_substitutions_clean.to_csv(
    f"{PROCESSED_PATH}fact_substitutions_clean.csv", index=False)
print(f"✅ fact_substitutions: {original:,} → {len(df):,} rows")

CLEANING: fact_order_items
----------------------------------------
   ✅ All order_id FKs valid
✅ fact_order_items: 2,718,037 → 2,718,037 rows

CLEANING: fact_picker_activity
----------------------------------------
   ✅ All picker_id FKs valid
✅ fact_picker_activity: 412,015 → 412,015 rows

CLEANING: fact_substitutions
----------------------------------------
   ✅ No self-substitutions found
   ✅ All order_id FKs valid
✅ fact_substitutions: 54,396 → 54,396 rows


In [10]:
# ============================================================
# FINAL CLEANING AUDIT REPORT
# ============================================================

cleaning_log_df = pd.DataFrame(cleaning_log)
cleaning_log_df.to_csv(
    f"{PROCESSED_PATH}cleaning_audit_log.csv", index=False)

print("=" * 65)
print("ZAPKART DATA CLEANING — FULL AUDIT REPORT")
print("=" * 65)
print(cleaning_log_df.to_string(index=False))

print("\n" + "=" * 65)
print("FINAL PROCESSED DATASET SUMMARY")
print("=" * 65)

cleaned_tables = {
    'dim_stores_clean':           dim_stores_clean,
    'dim_skus_clean':             dim_skus_clean,
    'dim_customers_clean':        dim_customers_clean,
    'dim_pickers_clean':          dim_pickers_clean,
    'fact_orders_clean':          fact_orders_clean,
    'fact_order_items_clean':     fact_order_items_clean,
    'fact_picker_activity_clean': fact_picker_activity_clean,
    'fact_substitutions_clean':   fact_substitutions_clean
}

for name, df in cleaned_tables.items():
    print(f"  {name:<40} {len(df):>10,} rows")

print("=" * 65)
print("✅ All 8 cleaned files saved to 01_Data/Processed/")
print("✅ Cleaning audit log saved")
print("\n All done. Ready for Step 4.")

ZAPKART DATA CLEANING — FULL AUDIT REPORT
        table                                   issue_found                                  action_taken  rows_affected
   dim_stores                         Added derived columns           store_age_days, cost_per_sqft added             10
     dim_skus                         Derived columns added gross_margin_pct_actual, is_high_margin added            499
dim_customers                     Extreme basket size > 985                     Capped at 99th percentile            500
dim_customers                              Duplicate emails                         Kept first occurrence           2390
dim_customers                         Derived columns added                             tenure_days added          47610
  dim_pickers Recalculated experience_months from join_date                     experience_months updated             80
  fact_orders       Delivery time discrepancy in 79875 rows                  Recalculated from components      